In [1]:
# !python -c "import cryptography; print(cryptography.__version__)"
# !python -c "from google.cloud import storage; print('google-cloud-storage import ok')"

In [5]:
!nvidia-smi

Mon Mar 16 12:58:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.65.06              Driver Version: 580.65.06      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:1B:00.0 Off |                    0 |
| N/A   32C    P0            114W /  700W |   61365MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [6]:
import dataclasses
import datetime
import functools
import math
import re
from typing import Optional

import cartopy.crs as ccrs
from google.cloud import storage
from graphcast import autoregressive
from graphcast import casting
from graphcast import checkpoint
from graphcast import data_utils
from graphcast import graphcast
from graphcast import normalization
from graphcast import rollout
from graphcast import xarray_jax
from graphcast import xarray_tree
from IPython.display import HTML
import ipywidgets as widgets
import haiku as hk
import jax
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
import numpy as np
import xarray




/home/apaudel/envs/env_graphcast/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [7]:
from google.cloud import storage

MODEL_KEY = "operational"
RES_KEY = "0.25"
LEVELS_KEY = "13"

client = storage.Client.create_anonymous_client()
bucket = client.bucket("dm_graphcast")

# Find checkpoint
param_matches = []
for blob in bucket.list_blobs(prefix="graphcast/params/"):
    name = blob.name.split("/")[-1]
    lname = name.lower()
    if (
        MODEL_KEY in lname
        and f"resolution {RES_KEY}" in lname
        and f"pressure levels {LEVELS_KEY}" in lname
        and name.endswith(".npz")
    ):
        param_matches.append(blob.name)

# Find stats
stats_matches = sorted(
    blob.name
    for blob in bucket.list_blobs(prefix="graphcast/stats/")
    if blob.name.endswith(".nc")
)

# Find smallest matching dataset for first inference
dataset_matches = []
for blob in bucket.list_blobs(prefix="graphcast/dataset/"):
    name = blob.name.split("/")[-1].lower()
    if (
        "source-era5" in name
        and f"res-{RES_KEY}" in name
        and f"levels-{LEVELS_KEY}" in name
        and name.endswith(".nc")
    ):
        dataset_matches.append(blob.name)

def extract_steps(path: str) -> int:
    name = path.split("/")[-1]
    marker = "_steps-"
    if marker not in name:
        return 10**9
    s = name.split(marker, 1)[1].split(".nc", 1)[0]
    return int(s)

dataset_matches = sorted(dataset_matches, key=extract_steps)

print("Checkpoint candidates:")
for p in param_matches:
    print(" ", p)

print("\nStats candidates:")
for s in stats_matches:
    print(" ", s)

print("\nDataset candidates:")
for d in dataset_matches:
    print(" ", d)

if len(param_matches) != 1:
    raise RuntimeError(f"Expected exactly 1 checkpoint match, found {len(param_matches)}")

if len(stats_matches) == 0:
    raise RuntimeError("No stats files found")

if len(dataset_matches) == 0:
    raise RuntimeError("No matching dataset files found")



Checkpoint candidates:
  graphcast/params/GraphCast_operational - ERA5-HRES 1979-2021 - resolution 0.25 - pressure levels 13 - mesh 2to6 - precipitation output only.npz

Stats candidates:
  graphcast/stats/diffs_stddev_by_level.nc
  graphcast/stats/mean_by_level.nc
  graphcast/stats/stddev_by_level.nc

Dataset candidates:
  graphcast/dataset/source-era5_date-2022-01-01_res-0.25_levels-13_steps-01.nc
  graphcast/dataset/source-era5_date-2022-01-01_res-0.25_levels-13_steps-04.nc
  graphcast/dataset/source-era5_date-2022-01-01_res-0.25_levels-13_steps-12.nc


In [8]:
selected_checkpoint = param_matches[0]
selected_stats = stats_matches
selected_dataset = dataset_matches[1]   # smallest steps-* file for first test

print("\nSelected files for download:")
print("CHECKPOINT:")
print(" ", selected_checkpoint)

print("\nSTATS:")
for s in selected_stats:
    print(" ", s)

print("\nDATASET:")
print(" ", selected_dataset)


Selected files for download:
CHECKPOINT:
  graphcast/params/GraphCast_operational - ERA5-HRES 1979-2021 - resolution 0.25 - pressure levels 13 - mesh 2to6 - precipitation output only.npz

STATS:
  graphcast/stats/diffs_stddev_by_level.nc
  graphcast/stats/mean_by_level.nc
  graphcast/stats/stddev_by_level.nc

DATASET:
  graphcast/dataset/source-era5_date-2022-01-01_res-0.25_levels-13_steps-04.nc


In [6]:
# from pathlib import Path

# OUTDIR = Path("/raid/apaudel/graphcast")
# OUTDIR.mkdir(parents=True, exist_ok=True)

# for remote_path in files_to_download:
#     rel = remote_path.replace("graphcast/", "", 1)
#     local_path = OUTDIR / rel
#     local_path.parent.mkdir(parents=True, exist_ok=True)

#     print(f"Downloading: {remote_path}")
#     print(f" -> {local_path}")

#     bucket.blob(remote_path).download_to_filename(str(local_path))

# print("\nDone.")

In [11]:
remote_path = selected_dataset
local_path = "/raid/apaudel/graphcast/dataset/source-era5_date-2022-01-01_res-0.25_levels-13_steps-04.nc"
bucket.blob(remote_path).download_to_filename(str(local_path))

In [9]:
from pathlib import Path

root = Path("/raid/apaudel/graphcast")
for sub in ["params", "stats", "dataset"]:
    print(f"\n{sub}:")
    for p in sorted((root / sub).glob("*")):
        print(" ", p.name)


params:
  GraphCast_operational - ERA5-HRES 1979-2021 - resolution 0.25 - pressure levels 13 - mesh 2to6 - precipitation output only.npz

stats:
  diffs_stddev_by_level.nc
  mean_by_level.nc
  stddev_by_level.nc

dataset:
  source-era5_date-2022-01-01_res-0.25_levels-13_steps-01.nc


In [8]:

ROOT = Path("/raid/apaudel/graphcast")

PARAMS_PATH = ROOT / "params" / "GraphCast_operational - ERA5-HRES 1979-2021 - resolution 0.25 - pressure levels 13 - mesh 2to6 - precipitation output only.npz"
DATASET_PATH = ROOT / "dataset" / "source-era5_date-2022-01-01_res-0.25_levels-13_steps-01.nc"

DIFFS_STDDEV_PATH = ROOT / "stats" / "diffs_stddev_by_level.nc"
MEAN_PATH = ROOT / "stats" / "mean_by_level.nc"
STDDEV_PATH = ROOT / "stats" / "stddev_by_level.nc"

print(PARAMS_PATH)
print(DATASET_PATH)
print(DIFFS_STDDEV_PATH)
print(MEAN_PATH)
print(STDDEV_PATH)

/raid/apaudel/graphcast/params/GraphCast_operational - ERA5-HRES 1979-2021 - resolution 0.25 - pressure levels 13 - mesh 2to6 - precipitation output only.npz
/raid/apaudel/graphcast/dataset/source-era5_date-2022-01-01_res-0.25_levels-13_steps-01.nc
/raid/apaudel/graphcast/stats/diffs_stddev_by_level.nc
/raid/apaudel/graphcast/stats/mean_by_level.nc
/raid/apaudel/graphcast/stats/stddev_by_level.nc


In [9]:
from graphcast import graphcast
from graphcast import checkpoint

with open(PARAMS_PATH, "rb") as f:
    ckpt = checkpoint.load(f, graphcast.CheckPoint)

params = ckpt.params
state = {}

model_config = ckpt.model_config
task_config = ckpt.task_config

print("Description:")
print(ckpt.description)
print("\nLicense:")
print(ckpt.license)

print("\nModel config:")
print(model_config)

print("\nTask config:")
print(task_config)

print("\nInput variables:")
print(task_config.input_variables)

print("\nTarget variables:")
print(task_config.target_variables)

print("\nForcing variables:")
print(task_config.forcing_variables)

print("\nPressure levels:")
print(task_config.pressure_levels)
print("Number of pressure levels:", len(task_config.pressure_levels))

Description:

GraphCast model at 0.25deg resolution, with 13 pressure levels. This model is
trained on ERA5 data from 1979 to 2017, and fine-tuned on HRES-fc0 data from
2016 to 2021 and can be causally evaluated on 2022 and later years. This model
does not take `total_precipitation_6hr` as inputs and can make predictions in an
operational setting (i.e., initialised from HRES-fc0).


License:

The model weights are licensed under the Creative Commons
Attribution-NonCommercial-ShareAlike 4.0 International (CC BY-NC-SA 4.0). You
may obtain a copy of the License at:
https://creativecommons.org/licenses/by-nc-sa/4.0/.
The weights were trained on ERA5 data, see README for attribution statement.


Model config:
ModelConfig(resolution=0.25, mesh_size=6, latent_size=512, gnn_msg_steps=16, hidden_layers=1, radius_query_fraction_edge_length=0.5999912857713345, mesh2grid_edge_normalization_factor=0.6180338738074472)

Task config:
TaskConfig(input_variables=('2m_temperature', 'mean_sea_level_pressu

In [10]:
!which python

/home/apaudel/envs/env_graphcast/bin/python


In [11]:
!pip list | grep -i graphcast

graphcast                 0.2.0.dev0  /home/apaudel/PROJECTS/graphcast/graphcast_fork_ap


In [12]:
import xarray

diffs_stddev_by_level = xarray.load_dataset(DIFFS_STDDEV_PATH).compute()
mean_by_level = xarray.load_dataset(MEAN_PATH).compute()
stddev_by_level = xarray.load_dataset(STDDEV_PATH).compute()

print("diffs_stddev_by_level:")
print(diffs_stddev_by_level)

print("\nmean_by_level:")
print(mean_by_level)

print("\nstddev_by_level:")
print(stddev_by_level)

diffs_stddev_by_level:
<xarray.Dataset> Size: 2kB
Dimensions:                           (level: 37)
Coordinates:
  * level                             (level) int32 148B 1 2 3 ... 950 975 1000
Data variables: (12/25)
    geopotential                      (level) float64 296B 1.153e+03 ... 210.3
    specific_humidity                 (level) float64 296B 6.64e-08 ... 0.000...
    temperature                       (level) float64 296B 3.68 3.832 ... 1.86
    u_component_of_wind               (level) float64 296B 8.713 7.032 ... 2.438
    v_component_of_wind               (level) float64 296B 9.788 7.719 ... 2.72
    vertical_velocity                 (level) float64 296B 0.001359 ... 0.08681
    ...                                ...
    year_progress                     float64 8B 0.0247
    year_progress_sin                 float64 8B 0.003034
    year_progress_cos                 float64 8B 0.003047
    day_progress                      float64 8B 0.433
    day_progress_sin             

In [13]:
for name, ds in {
    "diffs_stddev_by_level": diffs_stddev_by_level,
    "mean_by_level": mean_by_level,
    "stddev_by_level": stddev_by_level,
}.items():
    print(f"\n{name}")
    print("dims:", ds.dims)
    print("coords:", list(ds.coords))
    print("data_vars:", list(ds.data_vars))
    if "level" in ds.coords:
        print("levels:", ds["level"].values)


diffs_stddev_by_level
dims: FrozenMappingWarningOnValuesAccess({'level': 37})
coords: ['level']
data_vars: ['geopotential', 'specific_humidity', 'temperature', 'u_component_of_wind', 'v_component_of_wind', 'vertical_velocity', 'seconds_since_epoch', '10m_u_component_of_wind', '10m_v_component_of_wind', '2m_temperature', 'mean_sea_level_pressure', 'sea_ice_cover', 'sea_surface_temperature', 'surface_pressure', 'toa_incident_solar_radiation', 'toa_incident_solar_radiation_6hr', 'total_cloud_cover', 'total_column_water_vapour', 'total_precipitation_6hr', 'year_progress', 'year_progress_sin', 'year_progress_cos', 'day_progress', 'day_progress_sin', 'day_progress_cos']
levels: [   1    2    3    5    7   10   20   30   50   70  100  125  150  175
  200  225  250  300  350  400  450  500  550  600  650  700  750  775
  800  825  850  875  900  925  950  975 1000]

mean_by_level
dims: FrozenMappingWarningOnValuesAccess({'level': 37})
coords: ['level']
data_vars: ['geopotential', 'specific_hu

In [14]:
example_batch = xarray.load_dataset(DATASET_PATH).compute()

print(example_batch)

/tmp/ipykernel_143181/1842553982.py:1: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  example_batch = xarray.load_dataset(DATASET_PATH).compute()


<xarray.Dataset> Size: 1GB
Dimensions:                       (lon: 1440, lat: 721, level: 13, time: 3,
                                   batch: 1)
Coordinates:
  * lon                           (lon) float32 6kB 0.0 0.25 0.5 ... 359.5 359.8
  * lat                           (lat) float32 3kB -90.0 -89.75 ... 89.75 90.0
  * level                         (level) int32 52B 50 100 150 ... 850 925 1000
  * time                          (time) timedelta64[ns] 24B 00:00:00 ... 12:...
    datetime                      (batch, time) datetime64[ns] 24B 2022-01-01...
Dimensions without coordinates: batch
Data variables: (12/14)
    geopotential_at_surface       (lat, lon) float32 4MB 2.735e+04 ... -0.07617
    land_sea_mask                 (lat, lon) float32 4MB 1.0 1.0 1.0 ... 0.0 0.0
    2m_temperature                (batch, time, lat, lon) float32 12MB 250.7 ...
    mean_sea_level_pressure       (batch, time, lat, lon) float32 12MB 9.931e...
    10m_v_component_of_wind       (batch, time, lat

In [15]:
print("Dimensions:")
print(example_batch.dims)

print("\nCoordinates:")
print(list(example_batch.coords))

print("\nData variables:")
print(list(example_batch.data_vars))

if "time" in example_batch.coords:
    print("\nTime values:")
    print(example_batch["time"].values)
    print("Number of timesteps:", example_batch.sizes["time"])

if "level" in example_batch.coords:
    print("\nPressure levels:")
    print(example_batch["level"].values)
    print("Number of pressure levels:", example_batch.sizes["level"])

if "lat" in example_batch.coords:
    print("\nlat size:", example_batch.sizes["lat"])

if "lon" in example_batch.coords:
    print("lon size:", example_batch.sizes["lon"])

Dimensions:
FrozenMappingWarningOnValuesAccess({'lon': 1440, 'lat': 721, 'level': 13, 'time': 3, 'batch': 1})

Coordinates:
['lon', 'lat', 'level', 'time', 'datetime']

Data variables:
['geopotential_at_surface', 'land_sea_mask', '2m_temperature', 'mean_sea_level_pressure', '10m_v_component_of_wind', '10m_u_component_of_wind', 'total_precipitation_6hr', 'toa_incident_solar_radiation', 'temperature', 'geopotential', 'u_component_of_wind', 'v_component_of_wind', 'vertical_velocity', 'specific_humidity']

Time values:
[             0 21600000000000 43200000000000]
Number of timesteps: 3

Pressure levels:
[  50  100  150  200  250  300  400  500  600  700  850  925 1000]
Number of pressure levels: 13

lat size: 721
lon size: 1440


In [16]:
dataset_vars = set(example_batch.data_vars)
input_vars = set(task_config.input_variables)
target_vars = set(task_config.target_variables)
forcing_vars = set(task_config.forcing_variables)

print("Missing input vars:", sorted(input_vars - dataset_vars))
print("Missing target vars:", sorted(target_vars - dataset_vars))
print("Missing forcing vars:", sorted(forcing_vars - dataset_vars))

print("\nExtra dataset vars:", sorted(dataset_vars - (input_vars | target_vars | forcing_vars)))

Missing input vars: ['day_progress_cos', 'day_progress_sin', 'year_progress_cos', 'year_progress_sin']
Missing target vars: []
Missing forcing vars: ['day_progress_cos', 'day_progress_sin', 'year_progress_cos', 'year_progress_sin']

Extra dataset vars: []


In [17]:
import dataclasses
from graphcast import data_utils

train_inputs, train_targets, train_forcings = data_utils.extract_inputs_targets_forcings(
    example_batch,
    target_lead_times=slice("6h", "6h"),
    **dataclasses.asdict(task_config)
)

eval_inputs, eval_targets, eval_forcings = data_utils.extract_inputs_targets_forcings(
    example_batch,
    target_lead_times=slice("6h", "6h"),
    **dataclasses.asdict(task_config)
)

print("All Examples:  ", example_batch.dims.mapping)
print("Train Inputs:  ", train_inputs.dims.mapping)
print("Train Targets: ", train_targets.dims.mapping)
print("Train Forcings:", train_forcings.dims.mapping)
print("Eval Inputs:   ", eval_inputs.dims.mapping)
print("Eval Targets:  ", eval_targets.dims.mapping)
print("Eval Forcings: ", eval_forcings.dims.mapping)

print("\nTrain input vars:")
print(list(train_inputs.data_vars))

print("\nTrain target vars:")
print(list(train_targets.data_vars))

print("\nTrain forcing vars:")
print(list(train_forcings.data_vars))

All Examples:   {'lon': 1440, 'lat': 721, 'level': 13, 'time': 3, 'batch': 1}
Train Inputs:   {'batch': 1, 'time': 2, 'lat': 721, 'lon': 1440, 'level': 13}
Train Targets:  {'batch': 1, 'time': 1, 'lat': 721, 'lon': 1440, 'level': 13}
Train Forcings: {'batch': 1, 'time': 1, 'lat': 721, 'lon': 1440}
Eval Inputs:    {'batch': 1, 'time': 2, 'lat': 721, 'lon': 1440, 'level': 13}
Eval Targets:   {'batch': 1, 'time': 1, 'lat': 721, 'lon': 1440, 'level': 13}
Eval Forcings:  {'batch': 1, 'time': 1, 'lat': 721, 'lon': 1440}

Train input vars:
['2m_temperature', 'mean_sea_level_pressure', '10m_v_component_of_wind', '10m_u_component_of_wind', 'temperature', 'geopotential', 'u_component_of_wind', 'v_component_of_wind', 'vertical_velocity', 'specific_humidity', 'toa_incident_solar_radiation', 'year_progress_sin', 'year_progress_cos', 'day_progress_sin', 'day_progress_cos', 'geopotential_at_surface', 'land_sea_mask']

Train target vars:
['2m_temperature', 'mean_sea_level_pressure', '10m_v_component_o

In [18]:
diffs_stddev_by_level = xarray.load_dataset(DIFFS_STDDEV_PATH).compute()
mean_by_level = xarray.load_dataset(MEAN_PATH).compute()
stddev_by_level = xarray.load_dataset(STDDEV_PATH).compute()

In [19]:
import functools
import haiku as hk
import jax

from graphcast import autoregressive
from graphcast import casting
from graphcast import graphcast
from graphcast import normalization

def construct_wrapped_graphcast(
    model_config: graphcast.ModelConfig,
    task_config: graphcast.TaskConfig):
  predictor = graphcast.GraphCast(model_config, task_config)
  predictor = casting.Bfloat16Cast(predictor)
  predictor = normalization.InputsAndResiduals(
      predictor,
      diffs_stddev_by_level=diffs_stddev_by_level,
      mean_by_level=mean_by_level,
      stddev_by_level=stddev_by_level)
  predictor = autoregressive.Predictor(
      predictor,
      gradient_checkpointing=True)
  return predictor


@hk.transform_with_state
def run_forward(model_config, task_config, inputs, targets_template, forcings):
  predictor = construct_wrapped_graphcast(model_config, task_config)
  return predictor(inputs, targets_template=targets_template, forcings=forcings)


def with_configs(fn):
  return functools.partial(fn, model_config=model_config, task_config=task_config)

def with_params(fn):
  return functools.partial(fn, params=params, state=state)

def drop_state(fn):
  return lambda **kw: fn(**kw)[0]

run_forward_jitted = drop_state(
    with_params(
        jax.jit(with_configs(run_forward.apply))
    )
)

print("Inference function ready.")

Inference function ready.


In [44]:
def scale(
    data: xarray.Dataset,
    center: Optional[float] = None,
    robust: bool = False,
    ) -> tuple[xarray.Dataset, matplotlib.colors.Normalize, str]:
  vmin = np.nanpercentile(data, (2 if robust else 0))
  vmax = np.nanpercentile(data, (98 if robust else 100))
  if center is not None:
    diff = max(vmax - center, center - vmin)
    vmin = center - diff
    vmax = center + diff
  return (data, matplotlib.colors.Normalize(vmin, vmax),
          ("RdBu_r" if center is not None else "viridis"))



def select(
    data: xarray.Dataset,
    variable: str,
    level: Optional[int] = None,
    max_steps: Optional[int] = None
    ) -> xarray.Dataset:
  data = data[variable]
  if "batch" in data.dims:
    data = data.isel(batch=0)
  if max_steps is not None and "time" in data.sizes and max_steps < data.sizes["time"]:
    data = data.isel(time=range(0, max_steps))
  if level is not None and "level" in data.coords:
    data = data.sel(level=level)
  return data

def plot_data(
    data: dict[str, xarray.Dataset],
    fig_title: str,
    plot_size: float = 5,
    robust: bool = False,
    cols: int = 4
    ) -> tuple[xarray.Dataset, matplotlib.colors.Normalize, str]:

  first_data = next(iter(data.values()))[0]
  max_steps = first_data.sizes.get("time", 1)
  assert all(max_steps == d.sizes.get("time", 1) for d, _, _ in data.values())

  cols = min(cols, len(data))
  rows = math.ceil(len(data) / cols)
  figure = plt.figure(figsize=(plot_size * 2 * cols,
                               plot_size * rows))
  figure.suptitle(fig_title, fontsize=16)
  figure.subplots_adjust(wspace=0, hspace=0)
  figure.tight_layout()

  images = []
  for i, (title, (plot_data, norm, cmap)) in enumerate(data.items()):
    ax = figure.add_subplot(rows, cols, i+1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title)
    im = ax.imshow(
        plot_data.isel(time=0, missing_dims="ignore"), norm=norm,
        origin="lower", cmap=cmap)
    plt.colorbar(
        mappable=im,
        ax=ax,
        orientation="vertical",
        pad=0.02,
        aspect=16,
        shrink=0.75,
        cmap=cmap,
        extend=("both" if robust else "neither"))
    images.append(im)

  def update(frame):
    if "time" in first_data.dims:
      td = datetime.timedelta(microseconds=first_data["time"][frame].item() / 1000)
      figure.suptitle(f"{fig_title}, {td}", fontsize=16)
    else:
      figure.suptitle(fig_title, fontsize=16)
    for im, (plot_data, norm, cmap) in zip(images, data.values()):
      im.set_data(plot_data.isel(time=frame, missing_dims="ignore"))

  ani = animation.FuncAnimation(
      fig=figure, func=update, frames=max_steps, interval=250)
  plt.close(figure.number)
  return HTML(ani.to_jshtml())


In [30]:
import numpy as np
from graphcast import rollout

predictions = rollout.chunked_prediction(
    run_forward_jitted,
    rng=jax.random.PRNGKey(0),
    inputs=eval_inputs,
    targets_template=eval_targets * np.nan,
    forcings=eval_forcings,
)

predictions

/home/apaudel/PROJECTS/graphcast/graphcast_fork_ap/graphcast/rollout.py:295: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  num_target_steps = targets_template.dims["time"]


<xarray.Dataset> Size: 345MB
Dimensions:                  (time: 1, batch: 1, lat: 721, lon: 1440, level: 13)
Coordinates:
  * lon                      (lon) float32 6kB 0.0 0.25 0.5 ... 359.5 359.8
  * lat                      (lat) float32 3kB -90.0 -89.75 -89.5 ... 89.75 90.0
  * level                    (level) int32 52B 50 100 150 200 ... 850 925 1000
  * time                     (time) timedelta64[ns] 8B 06:00:00
Dimensions without coordinates: batch
Data variables:
    10m_u_component_of_wind  (time, batch, lat, lon) float32 4MB -0.3751 ... ...
    10m_v_component_of_wind  (time, batch, lat, lon) float32 4MB -1.151 ... -...
    2m_temperature           (time, batch, lat, lon) float32 4MB 248.6 ... 247.7
    geopotential             (time, batch, level, lat, lon) float32 54MB 1.98...
    mean_sea_level_pressure  (time, batch, lat, lon) float32 4MB 9.986e+04 .....
    specific_humidity        (time, batch, level, lat, lon) float32 54MB 2.90...
    temperature              (time, batch, level, lat, lon) float32 54MB 238....
    total_precipitation_6hr  (time, batch, lat, lon) float32 4MB 0.0003563 .....
    u_component_of_wind      (time, batch, level, lat, lon) float32 54MB 1.42...
    v_component_of_wind      (time, batch, level, lat, lon) float32 54MB -0.5...
    vertical_velocity        (time, batch, level, lat, lon) float32 54MB -0.0...

In [31]:
print(predictions)
print(predictions.dims)
print(list(predictions.data_vars))

<xarray.Dataset> Size: 345MB
Dimensions:                  (time: 1, batch: 1, lat: 721, lon: 1440, level: 13)
Coordinates:
  * lon                      (lon) float32 6kB 0.0 0.25 0.5 ... 359.5 359.8
  * lat                      (lat) float32 3kB -90.0 -89.75 -89.5 ... 89.75 90.0
  * level                    (level) int32 52B 50 100 150 200 ... 850 925 1000
  * time                     (time) timedelta64[ns] 8B 06:00:00
Dimensions without coordinates: batch
Data variables:
    10m_u_component_of_wind  (time, batch, lat, lon) float32 4MB -0.3751 ... ...
    10m_v_component_of_wind  (time, batch, lat, lon) float32 4MB -1.151 ... -...
    2m_temperature           (time, batch, lat, lon) float32 4MB 248.6 ... 247.7
    geopotential             (time, batch, level, lat, lon) float32 54MB 1.98...
    mean_sea_level_pressure  (time, batch, lat, lon) float32 4MB 9.986e+04 .....
    specific_humidity        (time, batch, level, lat, lon) float32 54MB 2.90...
    temperature              (time, b

In [45]:
# @title Choose predictions to plot

plot_pred_variable = widgets.Dropdown(
    options=predictions.data_vars.keys(),
    value="2m_temperature",
    description="Variable")
plot_pred_level = widgets.Dropdown(
    options=predictions.coords["level"].values,
    value=500,
    description="Level")
plot_pred_robust = widgets.Checkbox(value=True, description="Robust")
plot_pred_max_steps = widgets.IntSlider(
    min=1,
    max=predictions.dims["time"],
    value=predictions.dims["time"],
    description="Max steps")

widgets.VBox([
    plot_pred_variable,
    plot_pred_level,
    plot_pred_robust,
    plot_pred_max_steps,
    widgets.Label(value="Run the next cell to plot the predictions. Rerunning this cell clears your selection.")
])

/tmp/ipykernel_143181/3109806352.py:14: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  max=predictions.dims["time"],
/tmp/ipykernel_143181/3109806352.py:15: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  value=predictions.dims["time"],


In [55]:
# @title Choose data to plot

plot_example_variable = widgets.Dropdown(
    options=example_batch.data_vars.keys(),
    value="2m_temperature",
    description="Variable")
plot_example_level = widgets.Dropdown(
    options=example_batch.coords["level"].values,
    value=500,
    description="Level")
plot_example_robust = widgets.Checkbox(value=True, description="Robust")
plot_example_max_steps = widgets.IntSlider(
    min=1, max=example_batch.dims["time"], value=example_batch.dims["time"],
    description="Max steps")

widgets.VBox([
    plot_example_variable,
    plot_example_level,
    plot_example_robust,
    plot_example_max_steps,
    widgets.Label(value="Run the next cell to plot the data. Rerunning this cell clears your selection.")
])

/tmp/ipykernel_143181/2987701864.py:13: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  min=1, max=example_batch.dims["time"], value=example_batch.dims["time"],


In [56]:

plot_size = 7

data = {
    " ": scale(select(example_batch, plot_example_variable.value, plot_example_level.value, plot_example_max_steps.value),
              robust=plot_example_robust.value),
}
fig_title = plot_example_variable.value
if "level" in example_batch[plot_example_variable.value].coords:
  fig_title += f" at {plot_example_level.value} hPa"

plot_data(data, fig_title, plot_size, plot_example_robust.value)

In [57]:
# @title Plot predictions

plot_size = 5
plot_max_steps = min(predictions.dims["time"], plot_pred_max_steps.value)

data = {
    "Targets": scale(select(eval_targets, plot_pred_variable.value, plot_pred_level.value, plot_max_steps), robust=plot_pred_robust.value),
    "Predictions": scale(select(predictions, plot_pred_variable.value, plot_pred_level.value, plot_max_steps), robust=plot_pred_robust.value),
    "Diff": scale((select(eval_targets, plot_pred_variable.value, plot_pred_level.value, plot_max_steps) -
                        select(predictions, plot_pred_variable.value, plot_pred_level.value, plot_max_steps)),
                       robust=plot_pred_robust.value, center=0),
}
fig_title = plot_pred_variable.value
if "level" in predictions[plot_pred_variable.value].coords:
  fig_title += f" at {plot_pred_level.value} hPa"

plot_data(data, fig_title, plot_size, plot_pred_robust.value)


/tmp/ipykernel_143181/2833758296.py:4: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  plot_max_steps = min(predictions.dims["time"], plot_pred_max_steps.value)


In [3]:
import xarray
import numpy as np

def inspect_graphcast_example(nc_path: str) -> None:
    """Print everything about a GraphCast example .nc file.
    
    Run this on the Google-provided example input to understand the exact
    structure our ERA5 batch must match.
    """
    ds = xarray.open_dataset(nc_path)
    
    print("=" * 70)
    print("DIMENSIONS")
    print("=" * 70)
    for dim, size in ds.dims.items():
        print(f"  {dim}: {size}")

    print("\n" + "=" * 70)
    print("COORDINATES")
    print("=" * 70)
    for name, coord in ds.coords.items():
        print(f"\n  {name}")
        print(f"    dtype   : {coord.dtype}")
        print(f"    dims    : {coord.dims}")
        print(f"    shape   : {coord.shape}")
        if coord.size <= 10:
            print(f"    values  : {coord.values}")
        else:
            print(f"    first 5 : {coord.values[:5]}")
            print(f"    last  5 : {coord.values[-5:]}")

    print("\n" + "=" * 70)
    print("DATA VARIABLES")
    print("=" * 70)
    for name, var in ds.data_vars.items():
        print(f"\n  {name}")
        print(f"    dtype   : {var.dtype}")
        print(f"    dims    : {var.dims}")
        print(f"    shape   : {var.shape}")
        vals = var.values
        print(f"    min/max : {float(np.nanmin(vals)):.4g} / {float(np.nanmax(vals)):.4g}")
        print(f"    has NaN : {bool(np.isnan(vals).any())}")

    print("\n" + "=" * 70)
    print("GLOBAL ATTRIBUTES")
    print("=" * 70)
    if ds.attrs:
        for k, v in ds.attrs.items():
            print(f"  {k}: {v}")
    else:
        print("  (none)")

    print("\n" + "=" * 70)
    print("TIME COORDINATE DETAIL")
    print("=" * 70)
    if "time" in ds.coords:
        t = ds.coords["time"]
        print(f"  dtype  : {t.dtype}")
        print(f"  values : {t.values}")
    if "datetime" in ds.coords:
        dt = ds.coords["datetime"]
        print(f"\n  'datetime' coord exists!")
        print(f"  dtype  : {dt.dtype}")
        print(f"  dims   : {dt.dims}")
        print(f"  values : {dt.values}")
    else:
        print("\n  'datetime' coord: NOT PRESENT")

    print("\n" + "=" * 70)
    print("XARRAY REPR")
    print("=" * 70)
    print(ds)

In [4]:
inspect_graphcast_example(
    "/raid/apaudel/graphcast/dataset/source-era5_date-2022-01-01_res-0.25_levels-13_steps-01.nc"
)

/tmp/ipykernel_680908/2149884030.py:10: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds = xarray.open_dataset(nc_path)
/tmp/ipykernel_680908/2149884030.py:15: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  for dim, size in ds.dims.items():


DIMENSIONS
  lon: 1440
  lat: 721
  level: 13
  time: 3
  batch: 1

COORDINATES

  lon
    dtype   : float32
    dims    : ('lon',)
    shape   : (1440,)
    first 5 : [0.   0.25 0.5  0.75 1.  ]
    last  5 : [358.75 359.   359.25 359.5  359.75]

  lat
    dtype   : float32
    dims    : ('lat',)
    shape   : (721,)
    first 5 : [-90.   -89.75 -89.5  -89.25 -89.  ]
    last  5 : [89.   89.25 89.5  89.75 90.  ]

  level
    dtype   : int32
    dims    : ('level',)
    shape   : (13,)
    first 5 : [ 50 100 150 200 250]
    last  5 : [ 600  700  850  925 1000]

  time
    dtype   : timedelta64[ns]
    dims    : ('time',)
    shape   : (3,)
    values  : [             0 21600000000000 43200000000000]

  datetime
    dtype   : datetime64[ns]
    dims    : ('batch', 'time')
    shape   : (1, 3)
    values  : [['2022-01-01T00:00:00.000000000' '2022-01-01T06:00:00.000000000'
  '2022-01-01T12:00:00.000000000']]

DATA VARIABLES

  geopotential_at_surface
    dtype   : float32
    dims    : ('

In [ ]:
    print("inputs            dims:", dict(inputs.sizes))
    print("targets_template  dims:", dict(targets_template.sizes))
    print("forcings          dims:", dict(forcings.sizes))

In [12]:
inspect_graphcast_example(
    "/raid/apaudel/graphcast/dataset/source-era5_date-2022-01-01_res-0.25_levels-13_steps-04.nc"
)

/tmp/ipykernel_680908/2149884030.py:10: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds = xarray.open_dataset(nc_path)
/tmp/ipykernel_680908/2149884030.py:15: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  for dim, size in ds.dims.items():


DIMENSIONS
  lon: 1440
  lat: 721
  level: 13
  time: 6
  batch: 1

COORDINATES

  lon
    dtype   : float32
    dims    : ('lon',)
    shape   : (1440,)
    first 5 : [0.   0.25 0.5  0.75 1.  ]
    last  5 : [358.75 359.   359.25 359.5  359.75]

  lat
    dtype   : float32
    dims    : ('lat',)
    shape   : (721,)
    first 5 : [-90.   -89.75 -89.5  -89.25 -89.  ]
    last  5 : [89.   89.25 89.5  89.75 90.  ]

  level
    dtype   : int32
    dims    : ('level',)
    shape   : (13,)
    first 5 : [ 50 100 150 200 250]
    last  5 : [ 600  700  850  925 1000]

  time
    dtype   : timedelta64[ns]
    dims    : ('time',)
    shape   : (6,)
    values  : [              0  21600000000000  43200000000000  64800000000000
  86400000000000 108000000000000]

  datetime
    dtype   : datetime64[ns]
    dims    : ('batch', 'time')
    shape   : (1, 6)
    values  : [['2022-01-01T00:00:00.000000000' '2022-01-01T06:00:00.000000000'
  '2022-01-01T12:00:00.000000000' '2022-01-01T18:00:00.000000000'